In [2]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration


In [3]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [4]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [5]:
val_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [6]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [7]:
train_data.sample(10)

,id,dialogue,summary
6915,13728326-1,Mia: Hi Oscar. I’m Mia Lam. Patricia Johnson g...,Oscar and Mia are presenting in one panel tomo...
606,13716225,Alice: I never wanted a maternity photo shoot ...,Alice really likes a maternity photo shoot she...
11203,13828638,Jake: just finished watching the game\r\nDean:...,Jake and Dean are dissatisfied after watching ...
2978,13821643,Bill: Where should we go next? After NYC nothi...,"Bill, Donald and Lily are wondering which plac..."
3566,13829379,"Zeke: hey, im not a stalker - i swear! i got y...",Zeke got Kenya's number from Marianne. Zeke li...
8468,13716500,Beth: What kind of after school classes do you...,Lilly will take her kids to the after school c...
13399,13682146,Jenny: what did your dad say?\r\nLewis: he sai...,Lewis will meet Jenny at 4.30. Lewis will meet...
2378,13727798,Julia: Molly!!!! <3\r\nMolly: Yes!!!! So you a...,Julia is staying in Paris at least two weeks a...
9684,13829892,"Bob: hey, can you bring me the external drive ...",Ann needs to finish an e-mail. Then Ann will b...
5317,13729184,Lexi: What's that camera all the kids want thi...,Lexi asks Blake for advice about cameras. Blak...


In [8]:
train_data.shape

(14732, 3)

In [9]:
val_data.shape

(818, 3)

In [10]:
#Random Sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [11]:
train_data.shape

(4000, 3)

## Data Preprocessing

In [12]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) #lines"
    text = re.sub(r"\s+", " ", text) # Spaces
    text = re.sub(r"<.*?>", "", text) # HTML tags
    text = text.strip().lower()
    return text





In [13]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

In [14]:
val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [15]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:  claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

### Tokenize

In [16]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")


In [17]:
# Raw Data => Tokenized Inputs for fine-tuning the model
def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", truncation=True, max_length=512)
    targets = tokenizer(data["summary"], padding="max_length", truncation=True, max_length=150)

    inputs["labels"] = targets["input_ids"] # token ids => add to inputs as labels for training

    return inputs

In [18]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [19]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [20]:
# input_ids - dialogue => token ids
# 1 => EOS, 0 => padding
# attention_mask
# labels -  summary => token ids - target for training
# 1 -> valid token, 0 -> padding
len(train_dataset[0]["input_ids"])



512

In [21]:
len(train_dataset[0]["labels"])

150

In [22]:
len(train_dataset[0]["attention_mask"])

512

In [23]:
type(train_dataset[0])

transformers.tokenization_utils_base.BatchEncoding

In [24]:
type(train_dataset)

list

In [25]:
type(val_dataset)

list

## Working With Our Model

In [26]:
# NLP => Generation Task => T5 Model => Conditional Generation => T5ForConditionalGeneration => Based on the input, generate the output

model = T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [27]:
# Fine-tuning the model
import torch 

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device: ", device)
model.to(device)

Device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [28]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.14.0+cu130
CUDA available: True
CUDA version: 13.0
GPU count: 1
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [29]:
training_args = TrainingArguments(
    output_dir="./results",

    # Training
    num_train_epochs=10,
    learning_rate=5e-5,
    weight_decay=0.01,

    # Batch
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,

    # Evaluation & checkpoints
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    # Learning-rate warmup
    warmup_steps=500,

    # GPU optimization
    fp16=True,

    # Logging
    logging_steps=50,

    # Restore best checkpoint
    load_best_model_at_end=True,

    
)

In [30]:
from pathlib import Path

print("Current working directory:", Path.cwd())
print("Results directory:", Path("./results").resolve())

Current working directory: d:\AIML\projects\brief-sync
Results directory: D:\AIML\projects\brief-sync\results


In [31]:
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [32]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [33]:
# Train The Model
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.670051,0.509501
2,0.416336,0.375229
3,0.394731,0.362046
4,0.382074,0.355475
5,0.363275,0.352786
6,0.376942,0.349974
7,0.342354,0.348663
8,0.344172,0.348199
9,0.347648,0.347826
10,0.346260,0.347546


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2500, training_loss=1.0044151565551758, metrics={'train_runtime': 51572.3984, 'train_samples_per_second': 0.776, 'train_steps_per_second': 0.048, 'total_flos': 5413672058880000.0, 'train_loss': 1.0044151565551758, 'epoch': 10.0})

In [34]:
# model load => fine tune the model => save the model => load the model => inference

model.save_pretrained("./saved_summarizer_model")
tokenizer.save_pretrained("./saved_summarizer_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summarizer_model\\tokenizer_config.json',
 './saved_summarizer_model\\tokenizer.json')

In [36]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summarizer_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summarizer_model")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Test The Core Logic Of Summarization => Inference 


In [39]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # Clean - Preprocess the input dialogue

    # tokenize the input dialogue
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)



    # generate the summary using the fine-tuned model => token ids 
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,
        early_stopping = True
    )


    # token ids => decode => summary text  => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)
    return summary






In [40]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security and long-term societal impact.
